[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Indexes and Query Plans &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings and no
indexes, as the notebook's Setup did, and defines `show_plan`, and `open_copy`, which opens a fresh
copy of that database. Every task works on a copy of its own, so the tasks do not depend on one
another, and the last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
import time
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def open_copy(name):
    """A connection to a fresh copy of the database, with no indexes, at scratch/<name>.db."""
    return sqlite3.connect(shutil.copy(DATABASE, SCRATCH / f"{name}.db"))


def show_plan(conn, sql, parameters=()):
    """Print the plan SQLite chooses for a statement, one step to a line, indented under the step it belongs to."""
    depth = {0: -1}
    for step, parent, _, detail in conn.execute("EXPLAIN QUERY PLAN " + sql, parameters):
        depth[step] = depth[parent] + 1
        print("    " + "  " * depth[step] + detail)


print("built", DATABASE)


built scratch/stations.db


**1.** January, as `strftime` and as a range.


In [2]:
conn = open_copy("january")
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")

print("strftime:")
show_plan(conn, "SELECT hour, celsius FROM readings WHERE station_id = ? AND strftime('%Y-%m', hour) = ?",
          (ids["Tromso"], "2025-01"))
print("range:")
show_plan(conn, "SELECT hour, celsius FROM readings WHERE station_id = ? AND hour >= ? AND hour < ?",
          (ids["Tromso"], "2025-01-01", "2025-02-01"))
conn.close()


strftime:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
range:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour>? AND hour<?)


Both plans search the index for Tromso, and only the range goes on to narrow the search to January,
`hour>? AND hour<?`. With `strftime`, SQLite reads all 8,760 of Tromso's readings and works out the
month of each.


**2.** A minimum from the index alone.


In [3]:
conn = open_copy("minimum")
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
COLDEST = "SELECT MIN(celsius) FROM readings WHERE station_id = ?"

print("before:")
show_plan(conn, COLDEST, (ids["Svalbard"],))
conn.execute("CREATE INDEX readings_by_station_celsius ON readings (station_id, celsius)")
print("after:")
show_plan(conn, COLDEST, (ids["Svalbard"],))
print("coldest at Svalbard:", conn.execute(COLDEST, (ids["Svalbard"],)).fetchone()[0])
conn.close()


before:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
after:
    SEARCH readings USING COVERING INDEX readings_by_station_celsius (station_id=?)
coldest at Svalbard: -17.3


With only the index on `(station_id, hour)`, SQLite searches it for Svalbard, then reads every one
of the station's 8,760 rows to compare their temperatures. The index on `(station_id, celsius)`
holds both columns the query uses, and keeps a station's temperatures in order, so the plan is a
search of a covering index, and the minimum is the first entry after the `NULL`s.


**3.** One hour at every station, timed.


In [4]:
conn = open_copy("by_hour")
BY_HOUR = "SELECT station_id, celsius FROM readings WHERE hour = ?"
hours = [(datetime(2025, 1, 1) + timedelta(hours=n * 97 % 8760)).strftime("%Y-%m-%dT%H:%M") for n in range(100)]


def time_hours(conn):
    """Seconds taken to look up every hour in hours."""
    started = time.perf_counter()
    for hour in hours:
        conn.execute(BY_HOUR, (hour,)).fetchall()
    return time.perf_counter() - started


before = time_hours(conn)
conn.execute("CREATE INDEX readings_by_hour ON readings (hour)")
after = time_hours(conn)
print("100 lookups took more than 10 times as long before the index:", before > 10 * after)
conn.close()


100 lookups took more than 10 times as long before the index: True


Without the index, every lookup scanned 35,040 rows to find four, and with it, each one searched the
index for the hour. An index on `hour` alone serves this question, where the index on
`(station_id, hour)` does not, until `ANALYZE` lets SQLite plan a skip-scan.


**4.** An index on the month.


In [5]:
conn = open_copy("by_month")
conn.execute("CREATE INDEX readings_by_month ON readings (strftime('%m', hour))")

DECEMBER = "SELECT COUNT(*) FROM readings WHERE strftime('%m', hour) = ?"
show_plan(conn, DECEMBER, ("12",))
print("readings in December:", conn.execute(DECEMBER, ("12",)).fetchone()[0])
conn.close()


    SEARCH readings USING COVERING INDEX readings_by_month (<expr>=?)
readings in December: 2976


The condition names the same expression as the index, `strftime('%m', hour)`, so SQLite searches the
index for `'12'`, and since the count needs nothing else, the index covers the query. The expression
has to match the index's exactly: `strftime('%Y-%m', hour)` would scan.


**5.** Every index in the database.


In [6]:
conn = open_copy("index_list")
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
conn.execute("""
    CREATE TABLE notes (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL, day TEXT NOT NULL,
                        note TEXT NOT NULL, UNIQUE (station_id, day))
""")

for (table,) in conn.execute("SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name").fetchall():
    for _, name, unique, origin, partial in conn.execute(f'PRAGMA index_list("{table}")'):
        print(f"{table:<9} {name:<26} unique={unique} origin={origin} partial={partial}")
conn.close()


notes     sqlite_autoindex_notes_1   unique=1 origin=u partial=0
readings  readings_by_station_hour   unique=0 origin=c partial=0


`origin` says how an index came to be: `c` for `CREATE INDEX`, and `u` for the index SQLite built
for a `UNIQUE` constraint, named `sqlite_autoindex_notes_1`. A `PRIMARY KEY` that is not an
`INTEGER PRIMARY KEY` would show as `pk`. The table names come from `sqlite_schema`, never from
input.


**6.** A descending sort through the index.


In [7]:
conn = open_copy("latest")
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
LATEST = "SELECT hour, celsius FROM readings WHERE station_id = ? ORDER BY hour DESC LIMIT 5"

show_plan(conn, LATEST, (ids["Oslo"],))
print(conn.execute(LATEST, (ids["Oslo"],)).fetchall())
conn.close()


    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
[('2025-12-31T23:00', -3.3), ('2025-12-31T22:00', -2.9), ('2025-12-31T21:00', -2.4), ('2025-12-31T20:00', -1.9), ('2025-12-31T19:00', 0.2)]


There is no temporary B-tree: SQLite reads Oslo's part of the index backwards, from the last hour of
the year, and stops after five entries. An index serves a sort in either direction.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Indexes and Query Plans](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/15-indexes-and-query-plans.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
